In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    roc_curve, brier_score_loss,
)
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = Path('../data/processed')
FIG_DIR   = Path('../outputs/figures')
TABLE_DIR = Path('../outputs/tables')

plt.rcParams.update({
    'figure.dpi'        : 150,
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

MODEL_COLORS = {
    'Logistic Regression' : '#1F77B4',
    'Random Forest'       : '#2CA02C',
    'XGBoost'             : '#D62728',
}

INCOME_LABELS = {
    1: '<$10k',    2: '$10–15k',  3: '$15–20k',  4: '$20–25k',
    5: '$25–35k',  6: '$35–50k',  7: '$50–75k',  8: '$75–100k',
    9: '$100–150k',10: '$150–200k',11: '>$200k',
}

# 4-band grouping for stable within-bracket estimates
INCOME_BANDS = {
    'Low\n(<$25k)'      : [1, 2, 3, 4],
    'Mid-low\n($25–50k)': [5, 6],
    'Mid-high\n($50–150k)': [7, 8, 9],
    'High\n(>$150k)'    : [10, 11],
}

print('Imports OK')

Imports OK


In [6]:
df22 = pd.read_csv(DATA_DIR / 'brfss_2022_clean.csv')
df23 = pd.read_csv(DATA_DIR / 'brfss_2023_clean.csv')

for df in [df22, df23]:
    df['CHECKUP1'] = df['CHECKUP1'].replace({8.0: 5.0})

OUTCOME = 'COPD'
WEIGHT  = '_LLCPWT'

ORDINAL_FEATURES = [
    'INCOME3', 'EDUCA', 'SEXVAR', '_AGEG5YR', '_IMPRACE',
    'EMPLOY1', 'MARITAL', '_URBSTAT', '_SMOKER3', '_BMI5CAT',
    'PERSDOC3', 'CHECKUP1',
]
BINARY_FEATURES = [
    'EXERANY2', 'DRNKANY6', 'ASTHMA3', 'CVDINFR4', 'CVDCRHD4',
    'CVDSTRK3', 'HAVARTH3', 'CHCKDNY2', 'HLTHPLN2', 'MEDCOST1',
    'DIABETES', 'PREDIABETES',
]
ALL_FEATURES = ORDINAL_FEATURES + BINARY_FEATURES

# Surviving income interaction terms from notebook 06
INCOME_INT_COLS = [
    'INCOME3_x_EDUCA',
    'INCOME3_x_EXERANY2',
    'INCOME3_x__SMOKER3',
    'INCOME3_x_DIABETES',
    'INCOME3_x_CVDSTRK3',
    'INCOME3_x_DRNKANY6',
    'INCOME3_x_ASTHMA3',
    'INCOME3_x_MEDCOST1',
    'INCOME3_x__URBSTAT',
    'INCOME3_x_CVDCRHD4',
]

def add_income_interactions(df):
    df = df.copy()
    for col in INCOME_INT_COLS:
        parts = col.replace('INCOME3_x_', '').replace('_x_INCOME3', '')
        partner = parts
        df[col] = df['INCOME3'] * df[partner]
    return df

df22_int = add_income_interactions(df22)
df23_int = add_income_interactions(df23)

print(f'2022: {len(df22):,} rows')
print(f'2023: {len(df23):,} rows')
print(f'Interaction columns added: {len(INCOME_INT_COLS)}')

2022: 442,913 rows
2023: 431,257 rows
Interaction columns added: 10


In [7]:
n_neg = (df22[OUTCOME] == 0).sum()
n_pos = df22[OUTCOME].sum()
scale_pos_weight = n_neg / n_pos

def make_tree_pre(ordinal_cols, binary_cols):
    return ColumnTransformer(transformers=[
        ('ord', SimpleImputer(strategy='median'),        ordinal_cols),
        ('bin', SimpleImputer(strategy='most_frequent'), binary_cols),
    ], remainder='drop')

def make_lr_pre(ordinal_cols, binary_cols):
    return ColumnTransformer(transformers=[
        ('ord', Pipeline([
            ('imp',   SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), ordinal_cols),
        ('bin', SimpleImputer(strategy='most_frequent'), binary_cols),
    ], remainder='drop')

# ── Baseline models ───────────────────────────────────────────────────
BASELINE_MODELS = {
    'Logistic Regression': Pipeline([
        ('pre', make_lr_pre(ORDINAL_FEATURES, BINARY_FEATURES)),
        ('clf', LogisticRegression(
            class_weight='balanced', max_iter=1000,
            solver='saga', random_state=42, n_jobs=-1,
        )),
    ]),
    'Random Forest': Pipeline([
        ('pre', make_tree_pre(ORDINAL_FEATURES, BINARY_FEATURES)),
        ('clf', RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            max_features='sqrt', min_samples_leaf=50,
            random_state=42, n_jobs=-1,
        )),
    ]),
    'XGBoost': Pipeline([
        ('pre', make_tree_pre(ORDINAL_FEATURES, BINARY_FEATURES)),
        ('clf', xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            tree_method='hist', eval_metric='auc',
            random_state=42, n_jobs=-1, verbosity=0,
        )),
    ]),
}

# ── Interaction-augmented models ──────────────────────────────────────
INT_ORD = ORDINAL_FEATURES + INCOME_INT_COLS
INT_BIN = BINARY_FEATURES

INT_MODELS = {
    'Logistic Regression': Pipeline([
        ('pre', make_lr_pre(INT_ORD, INT_BIN)),
        ('clf', LogisticRegression(
            class_weight='balanced', max_iter=1000,
            solver='saga', random_state=42, n_jobs=-1,
        )),
    ]),
    'Random Forest': Pipeline([
        ('pre', make_tree_pre(INT_ORD, INT_BIN)),
        ('clf', RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            max_features='sqrt', min_samples_leaf=50,
            random_state=42, n_jobs=-1,
        )),
    ]),
    'XGBoost': Pipeline([
        ('pre', make_tree_pre(INT_ORD, INT_BIN)),
        ('clf', xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            tree_method='hist', eval_metric='auc',
            random_state=42, n_jobs=-1, verbosity=0,
        )),
    ]),
}

print('Fitting baseline models on full BRFSS 2022...')
X22      = df22[ALL_FEATURES]
X22_int  = df22_int[ALL_FEATURES + INCOME_INT_COLS]
y22      = df22[OUTCOME]

import time
for name, pipe in BASELINE_MODELS.items():
    t0 = time.time()
    pipe.fit(X22, y22)
    print(f'  Baseline {name}: {time.time()-t0:.1f}s')

print()
print('Fitting interaction-augmented models on full BRFSS 2022...')
for name, pipe in INT_MODELS.items():
    t0 = time.time()
    pipe.fit(X22_int, y22)
    print(f'  Interaction {name}: {time.time()-t0:.1f}s')

print('\nAll models fitted.')

Fitting baseline models on full BRFSS 2022...
  Baseline Logistic Regression: 7.6s
  Baseline Random Forest: 18.4s
  Baseline XGBoost: 2.6s

Fitting interaction-augmented models on full BRFSS 2022...
  Interaction Logistic Regression: 6.5s
  Interaction Random Forest: 22.7s
  Interaction XGBoost: 3.1s

All models fitted.


In [ ]:
# Use out-of-fold predictions for the fairness audit rather than
# in-sample predictions. In-sample predictions would be optimistic
# and unfair to compare across income brackets.

def get_oof_predictions(models_dict, X, y, n_splits=5):
    """Return out-of-fold predicted probabilities for each model."""
    cv  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = {name: np.zeros(len(y)) for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        for name, pipe in models_dict.items():
            pipe_clone = type(pipe)(**pipe.get_params())
            # Clone manually
            import copy
from sklearn.base import clone
            pipe_clone = copy.deepcopy(pipe)
            pipe_clone.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            oof[name][val_idx] = pipe_clone.predict_proba(
                X.iloc[val_idx])[:, 1]
        print(f'  Fold {fold+1}/5 done')

    return oof

print('Generating OOF predictions — baseline models...')
# Use out-of-fold predictions for the fairness audit rather than
# in-sample predictions. In-sample predictions would be optimistic
# and unfair to compare across income brackets.


def get_oof_predictions(models_dict, X, y, n_splits=5):
    """Return out-of-fold predicted probabilities for each model."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = {name: np.zeros(len(y)) for name in models_dict}

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        for name, pipe in models_dict.items():
            pipe_clone = clone(pipe)
            pipe_clone.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            oof[name][val_idx] = pipe_clone.predict_proba(X.iloc[val_idx])[:, 1]
        print(f'  Fold {fold+1}/5 done')

    return oof

print('Generating OOF predictions — baseline models...')
oof_baseline = get_oof_predictions(BASELINE_MODELS, X22, y22)

print()
print('Generating OOF predictions — interaction models...')
oof_int = get_oof_predictions(INT_MODELS, X22_int, y22)

print('\nOOF predictions complete.')
for name in BASELINE_MODELS:
    auc_b = roc_auc_score(y22, oof_baseline[name])
    auc_i = roc_auc_score(y22, oof_int[name])
    print(f'  {name:<22} Baseline AUC={auc_b:.4f}  Int AUC={auc_i:.4f}')

print()
print('Generating OOF predictions — interaction models...')
oof_int = get_oof_predictions(INT_MODELS, X22_int, y22)

print('\nOOF predictions complete.')
for name in BASELINE_MODELS:
    auc_b = roc_auc_score(y22, oof_baseline[name])
    auc_i = roc_auc_score(y22, oof_int[name])
    print(f'  {name:<22} Baseline AUC={auc_b:.4f}  Int AUC={auc_i:.4f}')

Generating OOF predictions — baseline models...


TypeError: Pipeline.__init__() got an unexpected keyword argument 'pre'

In [ ]:
# Compute one Youden-optimal threshold per model on the full OOF set.
# Apply this single threshold consistently across all income brackets.
# Using bracket-specific thresholds would be data leakage.

def youden_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

THRESHOLDS_BASE = {
    name: youden_threshold(y22, oof_baseline[name])
    for name in BASELINE_MODELS
}
THRESHOLDS_INT = {
    name: youden_threshold(y22, oof_int[name])
    for name in INT_MODELS
}

print('Youden-optimal thresholds:')
print(f'{"Model":<22}  {"Baseline":>10}  {"Interaction":>12}')
print('-' * 48)
for name in BASELINE_MODELS:
    print(f'{name:<22}  {THRESHOLDS_BASE[name]:>10.4f}  '
          f'{THRESHOLDS_INT[name]:>12.4f}')

In [ ]:
def bracket_metrics(y_true, y_prob, threshold):
    """Return sensitivity, specificity, PPV, FPR for one bracket."""
    if len(y_true) < 20 or y_true.sum() < 5:
        return None   # too few cases for stable estimate
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred,
                                       labels=[0, 1]).ravel()
    n_total    = len(y_true)
    n_copd     = y_true.sum()
    sens       = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec       = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv        = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    fpr        = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    pred_prev  = (tp + fp) / n_total
    actual_prev= n_copd / n_total
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    return {
        'n'           : n_total,
        'n_copd'      : int(n_copd),
        'actual_prev' : actual_prev * 100,
        'pred_prev'   : pred_prev * 100,
        'sensitivity' : sens,
        'specificity' : spec,
        'ppv'         : ppv,
        'fpr'         : fpr,
        'auc'         : auc,
    }

def run_bracket_audit(oof_probs, thresholds, df, y,
                      bracket_col='INCOME3'):
    """Return audit results by INCOME3 bracket for all models."""
    all_rows = []
    income_vals = sorted(df[bracket_col].dropna().unique())

    for name in oof_probs:
        thresh = thresholds[name]
        for inc_val in income_vals:
            mask = df[bracket_col] == inc_val
            if mask.sum() < 20:
                continue
            m = bracket_metrics(
                y[mask].values,
                oof_probs[name][mask.values],
                thresh
            )
            if m is None:
                continue
            m['model']       = name
            m['income_val']  = int(inc_val)
            m['income_label']= INCOME_LABELS.get(int(inc_val), str(int(inc_val)))
            all_rows.append(m)

    return pd.DataFrame(all_rows)

print('Running bracket-level audits...')
audit_baseline = run_bracket_audit(
    oof_baseline, THRESHOLDS_BASE,
    df22.reset_index(drop=True), y22.reset_index(drop=True)
)
audit_int = run_bracket_audit(
    oof_int, THRESHOLDS_INT,
    df22_int.reset_index(drop=True), y22.reset_index(drop=True)
)

print(f'Baseline audit rows : {len(audit_baseline)}')
print(f'Interaction audit rows: {len(audit_int)}')

In [ ]:
print('=== BASELINE MODEL — SENSITIVITY BY INCOME BRACKET ===\n')
print(f'{"Income":<14}', end='')
for name in BASELINE_MODELS:
    short = name.replace('Logistic Regression', 'LR').replace(
        'Random Forest', 'RF')
    print(f'  {short:>8} Sens  {short:>8} Spec', end='')
print()
print('-' * 80)

for inc_val in sorted(INCOME_LABELS.keys()):
    label = INCOME_LABELS[inc_val]
    row_str = f'{label:<14}'
    has_data = False
    for name in BASELINE_MODELS:
        sub = audit_baseline[
            (audit_baseline['model'] == name) &
            (audit_baseline['income_val'] == inc_val)
        ]
        if len(sub) == 0:
            row_str += f'  {"—":>12}  {"—":>12}'
        else:
            r = sub.iloc[0]
            row_str += (f'  {r["sensitivity"]:>12.3f}'
                        f'  {r["specificity"]:>12.3f}')
            has_data = True
    if has_data:
        print(row_str)

print()
print('=== SENSITIVITY GAP: LOWEST vs HIGHEST INCOME BRACKET ===\n')
for name in BASELINE_MODELS:
    sub = audit_baseline[audit_baseline['model'] == name].sort_values(
        'income_val')
    low_sens  = sub.iloc[0]['sensitivity']
    high_sens = sub.iloc[-1]['sensitivity']
    gap       = low_sens - high_sens
    print(f'  {name:<22}  Low: {low_sens:.3f}  High: {high_sens:.3f}  '
          f'Gap: {gap:+.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, name in zip(axes, BASELINE_MODELS.keys()):
    sub = audit_baseline[audit_baseline['model'] == name].sort_values(
        'income_val')
    colors = plt.cm.RdYlGn(
        np.linspace(0.15, 0.85, len(sub)))

    bars = ax.bar(range(len(sub)), sub['sensitivity'],
                  color=colors, alpha=0.9, width=0.65)

    # Highlight the gap
    overall_sens = sub['sensitivity'].mean()
    ax.axhline(overall_sens, color='#333', lw=1.5, ls='--',
               alpha=0.7, label=f'Mean: {overall_sens:.3f}')

    # Annotate lowest bracket
    ax.annotate(
        f'{sub.iloc[0]["sensitivity"]:.3f}',
        xy=(0, sub.iloc[0]['sensitivity']),
        xytext=(1.5, sub.iloc[0]['sensitivity'] - 0.04),
        fontsize=8, color='#D62728',
        arrowprops=dict(arrowstyle='->', color='#D62728', lw=1),
    )

    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub['income_label'], rotation=38,
                       ha='right', fontsize=8)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_ylabel('Sensitivity (recall)')
    ax.set_xlabel('Household Income Bracket')
    ax.set_ylim(0.4, 1.0)
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis='y', alpha=0.2)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x:.0%}'))

plt.suptitle('Baseline Model Sensitivity by Income Bracket — BRFSS 2022\n'
             'H4: Is sensitivity lower in lowest-income group?',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig17_sensitivity_by_income.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig17_sensitivity_by_income.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, name in zip(axes, BASELINE_MODELS.keys()):
    sub_b = audit_baseline[audit_baseline['model'] == name].sort_values(
        'income_val')
    sub_i = audit_int[audit_int['model'] == name].sort_values(
        'income_val')

    # Merge on income_val
    merged = sub_b[['income_val', 'income_label',
                     'sensitivity']].merge(
        sub_i[['income_val', 'sensitivity']],
        on='income_val', suffixes=('_base', '_int')
    )

    x = np.arange(len(merged))
    w = 0.35

    ax.bar(x - w/2, merged['sensitivity_base'], w,
           color=MODEL_COLORS[name], alpha=0.85, label='Baseline')
    ax.bar(x + w/2, merged['sensitivity_int'],  w,
           color=MODEL_COLORS[name], alpha=0.4,
           label='+ Income interactions')

    # Delta annotation on lowest bracket
    delta_low = (merged.iloc[0]['sensitivity_int'] -
                 merged.iloc[0]['sensitivity_base'])
    ax.annotate(f'Δ{delta_low:+.3f}',
                xy=(0 + w/2, merged.iloc[0]['sensitivity_int']),
                xytext=(0, 6), textcoords='offset points',
                ha='center', fontsize=8,
                color='green' if delta_low > 0 else '#D62728')

    ax.set_xticks(x)
    ax.set_xticklabels(merged['income_label'], rotation=38,
                       ha='right', fontsize=8)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_ylabel('Sensitivity')
    ax.set_ylim(0.4, 1.0)
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis='y', alpha=0.2)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'{x:.0%}'))

plt.suptitle('Sensitivity by Income: Baseline vs Interaction-Augmented\n'
             'Does adding INCOME3 interaction terms help lowest-income detection?',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig18_sensitivity_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig18_sensitivity_comparison.png')

In [ ]:
def band_metrics(oof_probs, thresholds, df, y):
    """Aggregate metrics into 4 income bands for stable estimates."""
    rows = []
    for band_label, inc_vals in INCOME_BANDS.items():
        mask = df['INCOME3'].isin(inc_vals)
        if mask.sum() < 50:
            continue
        for name in oof_probs:
            thresh = thresholds[name]
            m = bracket_metrics(
                y[mask].values,
                oof_probs[name][mask.values],
                thresh
            )
            if m is None:
                continue
            m['model']      = name
            m['band']       = band_label
            m['inc_vals']   = str(inc_vals)
            rows.append(m)
    return pd.DataFrame(rows)

bands_base = band_metrics(
    oof_baseline, THRESHOLDS_BASE,
    df22.reset_index(drop=True), y22.reset_index(drop=True)
)
bands_int = band_metrics(
    oof_int, THRESHOLDS_INT,
    df22_int.reset_index(drop=True), y22.reset_index(drop=True)
)

print('=== 4-BAND AUDIT SUMMARY — BASELINE MODELS ===\n')
print(f'{"Band":<22} {"Model":<22} {"n":>7} {"COPD n":>7} '
      f'{"Actual%":>8} {"Sens":>8} {"Spec":>8} {"AUC":>8}')
print('-' * 95)

band_order = list(INCOME_BANDS.keys())
for band in band_order:
    for name in BASELINE_MODELS:
        sub = bands_base[
            (bands_base['band'] == band) &
            (bands_base['model'] == name)
        ]
        if len(sub) == 0:
            continue
        r = sub.iloc[0]
        print(f'{band.replace(chr(10), " "):<22} {name:<22} '
              f'{r["n"]:>7,} {r["n_copd"]:>7,} '
              f'{r["actual_prev"]:>7.1f}% '
              f'{r["sensitivity"]:>8.3f} '
              f'{r["specificity"]:>8.3f} '
              f'{r["auc"]:>8.4f}')
    print()

bands_base.to_csv(TABLE_DIR / 'fairness_audit_bands_baseline.csv', index=False)
bands_int.to_csv(TABLE_DIR  / 'fairness_audit_bands_interaction.csv', index=False)
print('Saved fairness_audit_bands_baseline.csv')
print('Saved fairness_audit_bands_interaction.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (audit_df, thresh_dict, title) in zip(axes, [
    (audit_baseline, THRESHOLDS_BASE, 'Baseline Model'),
    (audit_int,      THRESHOLDS_INT,  '+ Income Interactions'),
]):
    models = list(BASELINE_MODELS.keys())
    inc_vals = sorted(audit_df['income_val'].unique())
    inc_labels = [INCOME_LABELS[v] for v in inc_vals]

    matrix = np.full((len(models), len(inc_vals)), np.nan)
    for i, name in enumerate(models):
        for j, inc_val in enumerate(inc_vals):
            sub = audit_df[
                (audit_df['model'] == name) &
                (audit_df['income_val'] == inc_val)
            ]
            if len(sub) > 0:
                matrix[i, j] = sub.iloc[0]['sensitivity']

    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn',
                   vmin=0.5, vmax=0.95)
    ax.set_xticks(range(len(inc_vals)))
    ax.set_xticklabels(inc_labels, rotation=38, ha='right', fontsize=8)
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(
        [n.replace('Logistic Regression', 'LR') for n in models],
        fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Income Bracket')

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=7,
                        color='white' if val < 0.65 else 'black')
    plt.colorbar(im, ax=ax, label='Sensitivity', fraction=0.046)

plt.suptitle('Sensitivity Heatmap by Income Bracket\n'
             'H4: Lowest-income group should show lowest sensitivity',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig19_sensitivity_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig19_sensitivity_heatmap.png')

In [ ]:
# Calibration check: does the model predict the right COPD prevalence
# in each income bracket? Systematic under-prediction in low income
# brackets = model is missing cases there.

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, name in zip(axes, BASELINE_MODELS.keys()):
    sub = audit_baseline[audit_baseline['model'] == name].sort_values(
        'income_val')
    x = np.arange(len(sub))
    w = 0.35

    ax.bar(x - w/2, sub['actual_prev'], w,
           color='#333333', alpha=0.85, label='Actual prevalence')
    ax.bar(x + w/2, sub['pred_prev'],   w,
           color=MODEL_COLORS[name], alpha=0.7, label='Predicted prevalence')

    ax.set_xticks(x)
    ax.set_xticklabels(sub['income_label'], rotation=38,
                       ha='right', fontsize=8)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_ylabel('COPD Prevalence (%)')
    ax.set_xlabel('Household Income Bracket')
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis='y', alpha=0.2)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.suptitle('Actual vs Predicted COPD Prevalence by Income Bracket\n'
             'Under-prediction in low-income brackets = systematic miss',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig20_calibration_by_income.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig20_calibration_by_income.png')

In [ ]:
audit_baseline['model_type'] = 'baseline'
audit_int['model_type']      = 'interaction'
full_audit = pd.concat([audit_baseline, audit_int], ignore_index=True)
full_audit.to_csv(TABLE_DIR / 'fairness_audit.csv', index=False)

print('=== NOTEBOOK 08 SUMMARY — H4 RESULT ===\n')

for name in BASELINE_MODELS:
    sub = audit_baseline[audit_baseline['model'] == name].sort_values(
        'income_val')
    sub_i = audit_int[audit_int['model'] == name].sort_values(
        'income_val')

    low_sens_b  = sub.iloc[0]['sensitivity']
    high_sens_b = sub.iloc[-1]['sensitivity']
    gap_b       = low_sens_b - high_sens_b

    low_sens_i  = sub_i.iloc[0]['sensitivity']
    gap_i       = low_sens_i - sub_i.iloc[-1]['sensitivity']
    improvement = low_sens_i - low_sens_b

    print(f'{name}:')
    print(f'  Lowest-income sensitivity  — baseline    : {low_sens_b:.3f}')
    print(f'  Highest-income sensitivity — baseline    : {high_sens_b:.3f}')
    print(f'  Sensitivity gap (low−high) — baseline    : {gap_b:+.3f}')
    print(f'  Lowest-income sensitivity  — interaction : {low_sens_i:.3f}')
    print(f'  Improvement in lowest bracket            : {improvement:+.3f}')
    print()

print('H4 interpretation:')
print('  Negative gap (low < high) → model misses more COPD in low-income group')
print('  Positive improvement → interaction terms partially correct this disparity')
print()
print('Saved:')
print('  fairness_audit.csv')
print('  fairness_audit_bands_baseline.csv')
print('  fairness_audit_bands_interaction.csv')
print()
print('Next: notebooks/09_temporal_validation.ipynb')